# ANNITIA Improved Baseline - Colab Notebook

This notebook is a self-contained Colab version of the improved ANNITIA baseline. It installs the needed packages, connects to your data, engineers longitudinal patient features, trains survival models for hepatic events and death, and saves `improved_submission.csv` to Google Drive.

Run the notebook from top to bottom. The heavier training cell can take a while because it performs local cross-validation and fits multiple survival models.


## Notebook Summary

Improved ANNITIA baseline.
What this changes compared with hello_world:
1. Replaces sparse raw repeated columns with longitudinal patient-level features.
2. Keeps endpoint-specific survival targets for hepatic event and death.
3. Trains Coxnet + Random Survival Forest, with optional XGBoost survival:cox.
4. Blends model predictions by ranks because the C-index only needs risk ordering.
Expected folder structure:
data/processed_wide/train.csv
data/processed_wide/val_test.csv
Output:
improved_submission.csv


## 1. Install Dependencies

Colab does not include `scikit-survival` by default. This cell installs the survival modeling package and XGBoost when the notebook is running in Colab. Local Jupyter runs skip the install step.


In [1]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install scikit-survival xgboost
else:
    print("Not running in Colab. Skipping pip install.")


## 2. Choose Data Location

Keep `USE_GOOGLE_DRIVE = True` when your project files are in Google Drive. The notebook expects your files in `/content/drive/MyDrive/MLComp` by default, but you can edit `DRIVE_PROJECT_DIR` if your folder has a different name.


In [6]:
from pathlib import Path
import os

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/MLComp")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PROJECT_DIR
else:
    PROJECT_DIR = Path.cwd()

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
else:
    print(f"Project directory does not exist yet: {PROJECT_DIR}")

print("Working directory:", Path.cwd())


Mounted at /content/drive
Working directory: /content/drive/MyDrive/MLComp


## 3. Optional Upload

Use this only if you are not using Google Drive. Set `USE_GOOGLE_DRIVE = False` above, then upload `train.csv` and `val_test.csv` or `test.csv` directly into the Colab runtime.


In [7]:
if IN_COLAB and not USE_GOOGLE_DRIVE:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Upload skipped. Using local/Drive files.")


Upload skipped. Using local/Drive files.


## 4.1. Imports

Load the libraries used by the baseline and detect whether optional XGBoost support is available.


In [8]:
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RepeatedStratifiedKFold, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False


## 4.2. Constants And Paths

Define default data paths, output path, random seed, target columns, and identifier columns. A later path-resolution cell overwrites the path values for Colab or local runs.


In [9]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
TRAIN_PATH = "data/processed_wide/train.csv"
TEST_PATH = "data/processed_wide/val_test.csv"
OUTPUT_PATH = "improved_submission.csv"

RANDOM_STATE = 27

TARGET_COLS = [
    "evenements_hepatiques_majeurs",
    "evenements_hepatiques_age_occur",
    "death",
    "death_age_occur",
]
ID_COLS = ["patient_id_anon", "trustii_id"]


## 4.3. Column Families

List the static patient fields and repeated visit measurements that are used to build features.


In [10]:
# ---------------------------------------------------------------------
# Column families from the real train.csv
# ---------------------------------------------------------------------
STATIC_CAT = ["gender", "T2DM", "Hypertension", "Dyslipidaemia", "bariatric_surgery"]
STATIC_NUM = ["bariatric_surgery_age"]

REPEATED_BASES = [
    "BMI",
    "alt",
    "ast",
    "bilirubin",
    "chol",
    "ggt",
    "gluc_fast",
    "plt",
    "triglyc",
    "fibrotest_BM_2",
    "aixp_aix_result_BM_3",
    "fibs_stiffness_med_BM_1",
]


## 4.4. Visit Column Helpers

Find and sort repeated visit columns like `Age_v1`, `Age_v2`, and lab-specific visit columns.


In [11]:
def get_age_cols(df: pd.DataFrame) -> list[str]:
    cols = [c for c in df.columns if re.fullmatch(r"Age_v\d+", c)]
    return sorted(cols, key=lambda c: int(c.rsplit("_v", 1)[1]))


def visit_cols(df: pd.DataFrame, base: str) -> list[str]:
    cols = [c for c in df.columns if re.fullmatch(fr"{re.escape(base)}_v\d+", c)]
    return sorted(cols, key=lambda c: int(c.rsplit("_v", 1)[1]))


## 4.5. Basic Longitudinal Helpers

Provide safe division and first/last non-null utilities for repeated measurements.


In [12]:
def safe_div(a, b):
    out = a / b
    if isinstance(out, pd.Series):
        out = out.replace([np.inf, -np.inf], np.nan)
    else:
        out = np.where(np.isfinite(out), out, np.nan)
    return out


def first_non_null(block: pd.DataFrame) -> pd.Series:
    return block.bfill(axis=1).iloc[:, 0]


def last_non_null(block: pd.DataFrame) -> pd.Series:
    return block.ffill(axis=1).iloc[:, -1]


def slope_against_age(block: pd.DataFrame, age_block: pd.DataFrame) -> pd.Series:
    """
    Row-wise OLS slope: value ~ age.
    Returns NaN if fewer than 2 aligned observations.
    """
    slopes = np.full(len(block), np.nan, dtype=float)

    for pos, idx in enumerate(block.index):
        y = block.loc[idx].to_numpy(dtype=float)
        x = age_block.loc[idx, age_block.columns[: len(block.columns)]].to_numpy(dtype=float)

        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() < 2:
            continue

        xv = x[valid]
        yv = y[valid]
        if np.allclose(xv, xv[0]):
            continue

        xc = xv - xv.mean()
        slopes[pos] = np.sum(xc * (yv - yv.mean())) / np.sum(xc ** 2)

    return pd.Series(slopes, index=block.index)


## 4.6. Visit Timing Helpers

Calculate per-patient slopes, last measurement ages, and measurement spans using the age columns.


In [13]:
def age_of_last_measure(block: pd.DataFrame, age_block: pd.DataFrame) -> pd.Series:
    out = np.full(len(block), np.nan, dtype=float)

    for pos, idx in enumerate(block.index):
        row = block.loc[idx]
        valid_cols = row.index[row.notna()]
        if len(valid_cols) == 0:
            continue
        last_col = valid_cols[-1]
        visit_num = int(last_col.rsplit("_v", 1)[1])
        age_col = f"Age_v{visit_num}"
        if age_col in age_block.columns:
            out[pos] = age_block.loc[idx, age_col]

    return pd.Series(out, index=block.index)


def span_years_for_measure(block: pd.DataFrame, age_block: pd.DataFrame) -> pd.Series:
    out = np.full(len(block), np.nan, dtype=float)

    for pos, idx in enumerate(block.index):
        row = block.loc[idx]
        valid_cols = row.index[row.notna()]
        if len(valid_cols) == 0:
            continue

        first_visit = int(valid_cols[0].rsplit("_v", 1)[1])
        last_visit = int(valid_cols[-1].rsplit("_v", 1)[1])

        first_age_col = f"Age_v{first_visit}"
        last_age_col = f"Age_v{last_visit}"

        if first_age_col in age_block.columns and last_age_col in age_block.columns:
            out[pos] = age_block.loc[idx, last_age_col] - age_block.loc[idx, first_age_col]

    return pd.Series(out, index=block.index)


## 4.7. Early Visit Features

Keep dense early visit values and add clinically useful ratios such as AST/ALT and FIB-4 per visit.


In [14]:
def add_visit_level_features(X: pd.DataFrame, df: pd.DataFrame, max_visit: int = 4) -> None:
    """
    Preserve raw dense early visits and add ratios/FIB-4 per early visit.
    Mutates X in place.
    """
    dense_bases = [
        "BMI", "alt", "ast", "bilirubin", "chol", "ggt", "gluc_fast",
        "plt", "triglyc", "fibrotest_BM_2", "aixp_aix_result_BM_3",
        "fibs_stiffness_med_BM_1",
    ]

    for v in range(1, max_visit + 1):
        age_col = f"Age_v{v}"

        for base in dense_bases:
            col = f"{base}_v{v}"
            if col in df.columns:
                X[col] = df[col]

        alt = df[f"alt_v{v}"] if f"alt_v{v}" in df.columns else np.nan
        ast = df[f"ast_v{v}"] if f"ast_v{v}" in df.columns else np.nan
        plt = df[f"plt_v{v}"] if f"plt_v{v}" in df.columns else np.nan
        ggt = df[f"ggt_v{v}"] if f"ggt_v{v}" in df.columns else np.nan
        age = df[age_col] if age_col in df.columns else np.nan

        if f"alt_v{v}" in df.columns and f"ast_v{v}" in df.columns:
            X[f"ast_alt_ratio_v{v}"] = safe_div(ast, alt)
        if f"ggt_v{v}" in df.columns and f"alt_v{v}" in df.columns:
            X[f"ggt_alt_ratio_v{v}"] = safe_div(ggt, alt)
        if f"ast_v{v}" in df.columns and f"plt_v{v}" in df.columns:
            X[f"ast_platelet_ratio_v{v}"] = safe_div(ast, plt)
        if all(c in df.columns for c in [age_col, f"ast_v{v}", f"alt_v{v}", f"plt_v{v}"]):
            X[f"fib4_v{v}"] = safe_div(age * ast, plt * np.sqrt(alt))


## 4.8. Patient Feature Engineering

Convert the wide repeated-visit table into dense patient-level features for model training and prediction.


In [15]:
def build_patient_features(df: pd.DataFrame, include_followup_observation_features: bool = True) -> pd.DataFrame:
    """
    Converts wide repeated-visit columns into dense patient-level features.

    include_followup_observation_features=True is leaderboard-friendly:
    it includes age_last_observed/followup/n_visits. This can be informative
    but may capture observation-process information.
    """
    X = pd.DataFrame(index=df.index)

    age_cols = get_age_cols(df)
    if len(age_cols) == 0:
        raise ValueError("No Age_v* columns found.")

    age_block = df[age_cols]

    # Static variables
    for col in STATIC_CAT + STATIC_NUM:
        if col in df.columns:
            X[col] = df[col]

    # Global age/follow-up features
    X["age_baseline"] = df["Age_v1"]

    if include_followup_observation_features:
        X["age_last_observed"] = age_block.max(axis=1)
        X["n_visits_age"] = age_block.notna().sum(axis=1)
        X["followup_years_observed"] = age_block.max(axis=1) - age_block.min(axis=1)

        if "bariatric_surgery_age" in df.columns:
            X["years_since_bariatric_at_baseline"] = df["Age_v1"] - df["bariatric_surgery_age"]
            X["years_since_bariatric_at_last"] = X["age_last_observed"] - df["bariatric_surgery_age"]

    # Longitudinal summaries
    for base in REPEATED_BASES:
        cols = visit_cols(df, base)
        if not cols:
            continue

        block = df[cols]

        X[f"{base}__first"] = first_non_null(block)
        X[f"{base}__last"] = last_non_null(block)
        X[f"{base}__min"] = block.min(axis=1)
        X[f"{base}__max"] = block.max(axis=1)
        X[f"{base}__mean"] = block.mean(axis=1)
        X[f"{base}__median"] = block.median(axis=1)
        X[f"{base}__std"] = block.std(axis=1)
        X[f"{base}__count"] = block.notna().sum(axis=1)
        X[f"{base}__miss_frac"] = block.isna().mean(axis=1)

        X[f"{base}__delta"] = X[f"{base}__last"] - X[f"{base}__first"]
        X[f"{base}__rel_change"] = safe_div(X[f"{base}__last"], X[f"{base}__first"]) - 1.0
        X[f"{base}__slope_per_year"] = slope_against_age(block, age_block)

        if include_followup_observation_features:
            X[f"{base}__age_last_measure"] = age_of_last_measure(block, age_block)
            X[f"{base}__span_years"] = span_years_for_measure(block, age_block)

    # Clinically meaningful derived features
    required = {"ast__first", "alt__first", "ast__last", "alt__last", "ggt__last", "plt__last"}
    if required.issubset(X.columns):
        X["ast_alt_ratio_first"] = safe_div(X["ast__first"], X["alt__first"])
        X["ast_alt_ratio_last"] = safe_div(X["ast__last"], X["alt__last"])
        X["ggt_alt_ratio_last"] = safe_div(X["ggt__last"], X["alt__last"])
        X["ast_platelet_ratio_last"] = safe_div(X["ast__last"], X["plt__last"])

    if {"age_baseline", "ast__first", "plt__first", "alt__first"}.issubset(X.columns):
        X["fib4_first"] = safe_div(
            X["age_baseline"] * X["ast__first"],
            X["plt__first"] * np.sqrt(X["alt__first"]),
        )

    age_for_last = X["age_last_observed"] if "age_last_observed" in X.columns else X["age_baseline"]
    if {"ast__last", "plt__last", "alt__last"}.issubset(X.columns):
        X["fib4_last"] = safe_div(
            age_for_last * X["ast__last"],
            X["plt__last"] * np.sqrt(X["alt__last"]),
        )

    # NIT burden / availability
    nit_last_cols = [
        c for c in [
            "fibrotest_BM_2__last",
            "aixp_aix_result_BM_3__last",
            "fibs_stiffness_med_BM_1__last",
        ]
        if c in X.columns
    ]
    if nit_last_cols:
        X["nit_available_count_last"] = X[nit_last_cols].notna().sum(axis=1)
        X["nit_last_rank_mean"] = X[nit_last_cols].rank(pct=True).mean(axis=1)

    # Metabolic interactions
    if "T2DM" in X.columns and "BMI__last" in X.columns:
        X["T2DM_x_BMI_last"] = X["T2DM"] * X["BMI__last"]
    if "T2DM" in X.columns and "gluc_fast__last" in X.columns:
        X["T2DM_x_glucose_last"] = X["T2DM"] * X["gluc_fast__last"]
    if "BMI__last" in X.columns and "triglyc__last" in X.columns:
        X["BMI_x_triglyc_last"] = X["BMI__last"] * X["triglyc__last"]

    add_visit_level_features(X, df, max_visit=4)

    # Clean infinities
    X = X.replace([np.inf, -np.inf], np.nan)

    # Keep only columns not completely empty
    X = X.dropna(axis=1, how="all")

    return X


## 4.9. Survival Target Preparation

Build endpoint-specific survival targets for hepatic events and death while handling censored observations.


In [16]:
# ---------------------------------------------------------------------
# Survival targets
# ---------------------------------------------------------------------
def prepare_survival_target(df: pd.DataFrame, outcome: str):
    df = df.copy()
    age_cols = get_age_cols(df)
    df["last_observed_age"] = df[age_cols].max(axis=1)

    if outcome == "hepatic":
        event_col = "evenements_hepatiques_majeurs"
        age_col = "evenements_hepatiques_age_occur"
        event_name = "Hepatic_event"

        invalid = (df[event_col] == 1) & df[age_col].isna()
        mask = ~invalid

    elif outcome == "death":
        event_col = "death"
        age_col = "death_age_occur"
        event_name = "Death"

        unknown = df[event_col].isna()
        invalid = (df[event_col] == 1) & df[age_col].isna()
        mask = ~unknown & ~invalid

    else:
        raise ValueError("outcome must be 'hepatic' or 'death'.")

    d = df.loc[mask].copy()
    event = (d[event_col] == 1).astype(bool).to_numpy()

    time = np.where(
        event,
        d[age_col].to_numpy(dtype=float) - d["Age_v1"].to_numpy(dtype=float),
        d["last_observed_age"].to_numpy(dtype=float) - d["Age_v1"].to_numpy(dtype=float),
    )
    time = np.maximum(time.astype(float), 1e-3)

    y = Surv.from_arrays(event=event, time=time, name_event=event_name, name_time="Time_years")
    return d, y, event, time


# ---------------------------------------------------------------------
# Preprocessing and models
# ---------------------------------------------------------------------


## 4.10. Feature Preprocessing

Create numeric and categorical preprocessing pipelines with imputation, scaling, and one-hot encoding.


In [17]:
def split_feature_types(X: pd.DataFrame):
    cat_cols = [c for c in STATIC_CAT if c in X.columns]
    num_cols = [c for c in X.columns if c not in cat_cols]
    return num_cols, cat_cols


def make_preprocessor(X: pd.DataFrame, sparse_threshold: float = 0.3):
    num_cols, cat_cols = split_feature_types(X)

    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median", add_indicator=True)),
        ("sc", StandardScaler(with_mean=False)),
    ])

    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="constant", fill_value=-1)),
        ("oh", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=sparse_threshold,
    )


## 4.11. Coxnet Model

Define the Coxnet survival model and alpha-selection cross-validation helper.


In [18]:
def make_coxnet_pipeline(X: pd.DataFrame) -> Pipeline:
    return Pipeline([
        ("pre", make_preprocessor(X, sparse_threshold=0.3)),
        ("cox", CoxnetSurvivalAnalysis(
            l1_ratio=0.9,
            alpha_min_ratio=0.01,
            max_iter=20000,
            fit_baseline_model=True,
        )),
    ])


def fit_coxnet_with_alpha_cv(X: pd.DataFrame, y, random_state: int = RANDOM_STATE) -> Pipeline:
    """
    Like the baseline, first fit an alpha path, then select the best alpha.
    This usually works better than leaving the full path inside predict().
    """
    path_model = make_coxnet_pipeline(X)
    path_model.fit(X, y)
    alphas = path_model.named_steps["cox"].alphas_

    cv = KFold(n_splits=5, shuffle=True, random_state=random_state)

    grid = GridSearchCV(
        estimator=Pipeline([
            ("pre", make_preprocessor(X, sparse_threshold=0.3)),
            ("cox", CoxnetSurvivalAnalysis(
                l1_ratio=0.9,
                max_iter=20000,
                fit_baseline_model=True,
            )),
        ]),
        param_grid={"cox__alphas": [[float(a)] for a in alphas]},
        cv=cv,
        error_score=0.5,
        n_jobs=-1,
    )
    grid.fit(X, y)
    return grid.best_estimator_


## 4.12. Random Survival Forest Model

Define the Random Survival Forest pipeline used as a nonlinear survival model.


In [19]:
def make_rsf_pipeline(X: pd.DataFrame) -> Pipeline:
    return Pipeline([
        ("pre", make_preprocessor(X, sparse_threshold=0.3)),
        ("rsf", RandomSurvivalForest(
            n_estimators=500,
            min_samples_leaf=20,
            min_samples_split=40,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


## 4.13. XGBoost Survival Model

Define the optional XGBoost `survival:cox` pipeline and its signed-time target convention.


In [20]:
def signed_time_label(event: np.ndarray, time: np.ndarray) -> np.ndarray:
    """
    XGBoost survival:cox convention:
    positive time = observed event, negative time = right-censored.
    """
    return np.where(event, time, -time)


def make_xgb_pipeline(X: pd.DataFrame) -> Pipeline:
    if not HAS_XGB:
        raise ImportError("xgboost is not installed.")

    return Pipeline([
        ("pre", make_preprocessor(X, sparse_threshold=1.0)),
        ("xgb", XGBRegressor(
            objective="survival:cox",
            n_estimators=600,
            learning_rate=0.025,
            max_depth=2,
            min_child_weight=10,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=5.0,
            reg_alpha=0.5,
            random_state=RANDOM_STATE,
            tree_method="hist",
            n_jobs=-1,
        )),
    ])


## 4.14. Cross-Validation Scoring

Compute concordance index scores for scikit-survival models and optional XGBoost models.


In [21]:
def cindex_from_y(y, pred: np.ndarray) -> float:
    event_name, time_name = y.dtype.names
    return float(concordance_index_censored(y[event_name], y[time_name], pred)[0])


def cv_cindex_sksurv(model_factory, X: pd.DataFrame, y, event: np.ndarray,
                    n_splits: int = 5, n_repeats: int = 3) -> tuple[float, float]:
    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
    )

    event_name, time_name = y.dtype.names
    scores = []

    for tr, va in cv.split(X, event.astype(int)):
        model = model_factory(X.iloc[tr])
        model.fit(X.iloc[tr], y[tr])
        pred = model.predict(X.iloc[va])
        scores.append(concordance_index_censored(y[event_name][va], y[time_name][va], pred)[0])

    return float(np.mean(scores)), float(np.std(scores))


def cv_cindex_xgb(X: pd.DataFrame, event: np.ndarray, time: np.ndarray,
                  n_splits: int = 5, n_repeats: int = 3) -> tuple[float, float]:
    if not HAS_XGB:
        return np.nan, np.nan

    cv = RepeatedStratifiedKFold(
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
    )

    scores = []
    y_signed = signed_time_label(event, time)

    for tr, va in cv.split(X, event.astype(int)):
        model = make_xgb_pipeline(X.iloc[tr])
        model.fit(X.iloc[tr], y_signed[tr])
        pred = model.predict(X.iloc[va])
        scores.append(concordance_index_censored(event[va], time[va], pred)[0])

    return float(np.mean(scores)), float(np.std(scores))


## 4.15. Prediction Blending

Rank-normalize model predictions and combine them into a blended risk score.


In [22]:
def rank_normalize(x) -> np.ndarray:
    return pd.Series(np.asarray(x)).rank(method="average", pct=True).to_numpy()


def blend_predictions(predictions: list[np.ndarray], weights: list[float] | None = None) -> np.ndarray:
    if weights is None:
        weights = [1.0] * len(predictions)

    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    blended = np.zeros(len(predictions[0]), dtype=float)
    for p, w in zip(predictions, weights):
        blended += w * rank_normalize(p)

    return blended


## 4.16. End-To-End Training Function

Run feature engineering, local CV, final model fitting, prediction blending, and submission export.


In [23]:
# ---------------------------------------------------------------------
# Main training
# ---------------------------------------------------------------------
def main():
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)

    print(f"train shape: {train_df.shape}")
    print(f"test shape : {test_df.shape}")

    X_all = build_patient_features(train_df, include_followup_observation_features=True)
    X_test = build_patient_features(test_df, include_followup_observation_features=True)

    # Align train/test feature columns. Missing columns are introduced as NaN.
    X_test = X_test.reindex(columns=X_all.columns)

    print(f"engineered feature count: {X_all.shape[1]}")
    print(f"xgboost available: {HAS_XGB}")

    # ---------------- Hepatic target ----------------
    hep_df, y_hep, hep_event, hep_time = prepare_survival_target(train_df, "hepatic")
    X_hep = X_all.loc[hep_df.index].copy()

    # ---------------- Death target ----------------
    death_df, y_death, death_event, death_time = prepare_survival_target(train_df, "death")
    X_death = X_all.loc[death_df.index].copy()

    print("\nTargets")
    print(f"hepatic: n={len(X_hep)}, events={hep_event.sum()} ({hep_event.mean():.3%})")
    print(f"death  : n={len(X_death)}, events={death_event.sum()} ({death_event.mean():.3%})")

    # ---------------- Local CV ----------------
    print("\nLocal CV on engineered features")
    hep_cox_cv = cv_cindex_sksurv(make_coxnet_pipeline, X_hep, y_hep, hep_event)
    death_cox_cv = cv_cindex_sksurv(make_coxnet_pipeline, X_death, y_death, death_event)

    print(f"Coxnet hepatic: mean={hep_cox_cv[0]:.4f}, std={hep_cox_cv[1]:.4f}")
    print(f"Coxnet death  : mean={death_cox_cv[0]:.4f}, std={death_cox_cv[1]:.4f}")
    print(f"Weighted Coxnet CV: {0.7 * hep_cox_cv[0] + 0.3 * death_cox_cv[0]:.4f}")

    if HAS_XGB:
        hep_xgb_cv = cv_cindex_xgb(X_hep, hep_event, hep_time)
        death_xgb_cv = cv_cindex_xgb(X_death, death_event, death_time)

        print(f"XGB hepatic   : mean={hep_xgb_cv[0]:.4f}, std={hep_xgb_cv[1]:.4f}")
        print(f"XGB death     : mean={death_xgb_cv[0]:.4f}, std={death_xgb_cv[1]:.4f}")
        print(f"Weighted XGB CV: {0.7 * hep_xgb_cv[0] + 0.3 * death_xgb_cv[0]:.4f}")

    # ---------------- Fit final models ----------------
    print("\nFitting final models")

    # Coxnet with alpha CV
    cox_hep = fit_coxnet_with_alpha_cv(X_hep, y_hep)
    cox_death = fit_coxnet_with_alpha_cv(X_death, y_death)

    pred_hep_list = [cox_hep.predict(X_test)]
    pred_death_list = [cox_death.predict(X_test)]
    weights = [0.45]

    # RSF
    rsf_hep = make_rsf_pipeline(X_hep)
    rsf_hep.fit(X_hep, y_hep)
    rsf_death = make_rsf_pipeline(X_death)
    rsf_death.fit(X_death, y_death)

    pred_hep_list.append(rsf_hep.predict(X_test))
    pred_death_list.append(rsf_death.predict(X_test))
    weights.append(0.25)

    # XGB survival:cox
    if HAS_XGB:
        xgb_hep = make_xgb_pipeline(X_hep)
        xgb_hep.fit(X_hep, signed_time_label(hep_event, hep_time))

        xgb_death = make_xgb_pipeline(X_death)
        xgb_death.fit(X_death, signed_time_label(death_event, death_time))

        pred_hep_list.append(xgb_hep.predict(X_test))
        pred_death_list.append(xgb_death.predict(X_test))
        weights.append(0.30)

    # Blend by ranks
    pred_hep = blend_predictions(pred_hep_list, weights)
    pred_death = blend_predictions(pred_death_list, weights)

    # ---------------- Submission ----------------
    id_col = "trustii_id" if "trustii_id" in test_df.columns else "patient_id_anon"

    submission = pd.DataFrame({
        "trustii_id": test_df[id_col].values,
        "risk_hepatic_event": pred_hep,
        "risk_death": pred_death,
    })

    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved {len(submission)} predictions -> {OUTPUT_PATH}")
    print(submission.head())


## 5. Resolve Input And Output Paths

This cell finds the training and validation/test CSVs in either the repo-style folder layout or a flat Colab upload layout. The final submission path is set to `improved_submission.csv` in the current project directory.


In [24]:
def find_first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError("Could not find any of these files:\n" + "\n".join(str(p) for p in paths))

TRAIN_PATH = str(find_first_existing([
    Path("data/processed_wide/train.csv"),
    Path("train.csv"),
    Path("/content/data/processed_wide/train.csv"),
    Path("/content/train.csv"),
]))

TEST_PATH = str(find_first_existing([
    Path("data/processed_wide/val_test.csv"),
    Path("val_test.csv"),
    Path("test.csv"),
    Path("/content/data/processed_wide/val_test.csv"),
    Path("/content/val_test.csv"),
    Path("/content/test.csv"),
]))

OUTPUT_PATH = str(Path.cwd() / "improved_submission.csv")

print("TRAIN_PATH :", TRAIN_PATH)
print("TEST_PATH  :", TEST_PATH)
print("OUTPUT    :", OUTPUT_PATH)
print("HAS_XGB   :", HAS_XGB)


TRAIN_PATH : train.csv
TEST_PATH  : test.csv
OUTPUT    : /content/drive/MyDrive/MLComp/improved_submission.csv
HAS_XGB   : True


## 6. Preview Data

Check the row/column counts and inspect the first rows before training. This is a quick sanity check that the notebook is reading the expected files.


In [25]:
train_preview = pd.read_csv(TRAIN_PATH)
test_preview = pd.read_csv(TEST_PATH)
print("train shape:", train_preview.shape)
print("test shape :", test_preview.shape)
display(train_preview.head())
display(test_preview.head())


train shape: (1253, 287)
test shape : (423, 284)


,patient_id_anon,gender,T2DM,Hypertension,Dyslipidaemia,bariatric_surgery,bariatric_surgery_age,Age_v1,Age_v2,Age_v3,...,fibs_stiffness_med_BM_1_v17,fibs_stiffness_med_BM_1_v18,fibs_stiffness_med_BM_1_v19,fibs_stiffness_med_BM_1_v20,fibs_stiffness_med_BM_1_v21,fibs_stiffness_med_BM_1_v22,evenements_hepatiques_majeurs,evenements_hepatiques_age_occur,death,death_age_occur
0,8MDD4V30T9NT,1,1.0,1.0,1.0,0.0,NaN,40.0,43.0,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0.0,NaN
1,3W5UZBIKCIDK,2,0.0,0.0,0.0,0.0,NaN,62.0,62.0,63.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0.0,NaN
2,9XUY41IBLJH7,1,0.0,0.0,1.0,0.0,NaN,48.0,51.0,53.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0.0,NaN
3,5LXO6QJIUJV6,1,1.0,1.0,1.0,0.0,NaN,55.0,56.0,58.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,67.0,1.0,72.0
4,1T2TALA753LC,1,0.0,0.0,0.0,0.0,NaN,45.0,48.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0.0,NaN


,trustii_id,patient_id_anon,gender,T2DM,Hypertension,Dyslipidaemia,bariatric_surgery,bariatric_surgery_age,Age_v1,Age_v2,...,fibs_stiffness_med_BM_1_v13,fibs_stiffness_med_BM_1_v14,fibs_stiffness_med_BM_1_v15,fibs_stiffness_med_BM_1_v16,fibs_stiffness_med_BM_1_v17,fibs_stiffness_med_BM_1_v18,fibs_stiffness_med_BM_1_v19,fibs_stiffness_med_BM_1_v20,fibs_stiffness_med_BM_1_v21,fibs_stiffness_med_BM_1_v22
0,1,WNNHJ7XVG0FN,2,0.0,0.0,1.0,0.0,NaN,35.0,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,53YKYYDU3ASN,2,0.0,0.0,0.0,2.0,NaN,30.0,30.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,K5J42OI6OAFN,2,0.0,1.0,1.0,0.0,NaN,68.0,69.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,YC3S0DPJHSG0,2,0.0,1.0,1.0,0.0,NaN,73.0,73.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,V6SGP3JK8XNL,1,1.0,1.0,1.0,2.0,NaN,55.0,56.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Train Models And Create Submission

This runs the full improved baseline: feature engineering, endpoint-specific survival target preparation, local CV, final model fitting, rank blending, and CSV export.


In [26]:
main()


train shape: (1253, 287)
test shape : (423, 284)
engineered feature count: 255
xgboost available: True

Targets
hepatic: n=1253, events=47 (3.751%)
death  : n=984, events=76 (7.724%)

Local CV on engineered features
Coxnet hepatic: mean=0.7308, std=0.1012
Coxnet death  : mean=0.9152, std=0.0369
Weighted Coxnet CV: 0.7861
XGB hepatic   : mean=0.7350, std=0.1033
XGB death     : mean=0.9190, std=0.0359
Weighted XGB CV: 0.7902

Fitting final models


KeyboardInterrupt: 

## 8. Save Submission To Google Drive

Read the generated CSV, save a copy to your Google Drive project folder, and display the first rows for a final check.


In [ ]:
submission = pd.read_csv(OUTPUT_PATH)

if IN_COLAB:
    drive_output_path = DRIVE_PROJECT_DIR / "improved_submission.csv"
    drive_output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(drive_output_path, index=False)
    print("Saved to Google Drive:", drive_output_path)
else:
    drive_output_path = Path(OUTPUT_PATH)
    print("Saved locally to:", drive_output_path)

print("submission shape:", submission.shape)
display(submission.head())


Saved to Google Drive: /content/drive/MyDrive/MLComp/improved_submission.csv
submission shape: (423, 3)


,trustii_id,risk_hepatic_event,risk_death
0,1,0.234634,0.193262
1,2,0.267612,0.409870
2,3,0.320804,0.412530
3,4,0.787589,0.972459
4,5,0.485579,0.563239
